# Day 1.2 — Configuring Model Behaviour

Previously, we sent one question and received unrestricted text. Now we use message roles and clear constraints.

```text
System instructions + user request → model → better-shaped text
```

The guided notebooks use the issued OpenRouter key. Optional Ollama and direct OpenAI setup remains in Notebook 01.

## Before you begin

### Learning outcomes

Separate standing instructions from the current task and observe temperature/output constraints.

Architecture reference: [D01](../../diagrams/source/day_01.md).

### Expected observation

The constrained response follows the requested format more reliably than the broad prompt.


## Learning objectives

Distinguish system and user messages, configure answer organization and length, and observe that instructions improve consistency without guaranteeing a schema.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key) if api_key else None
COURSE_MODEL=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
def ask(messages,max_tokens=400):
    if not client:
        constrained=any(message.get("role")=="system" for message in messages)
        return "Definition: An agent chooses bounded actions.\nExample: It requests a calculator tool.\nLimitation: Host code must control execution." if constrained else "An AI agent uses a model and tools to work toward a goal."
    return client.chat.completions.create(model=COURSE_MODEL,messages=messages,max_tokens=max_tokens,
        extra_body={"reasoning":{"effort":"low","exclude":True}}).choices[0].message.content
print("Route:","OpenRouter" if client else "mock fallback")


## Build: begin with a broad request and observe its variability

In [ ]:
print(ask([{"role": "user", "content": "Explain an AI agent."}]))

## Improve: separate standing instructions from the current task

A system message describes how the model should behave for the call. A user message contains the current request.

In [ ]:
messages = [
    {"role": "system", "content": (
        "You teach engineering students new to agentic AI. Use plain language. "
        "Give exactly three short sections: Definition, Example, and Limitation."
    )},
    {"role": "user", "content": "Explain an AI agent."},
]
print(ask(messages))

### Observe

Did all sections appear? Is it beginner-friendly? Does rerunning produce identical punctuation? Clear instructions help, but application boundaries still require validation.

## Break it

Ask for a Python dictionary with exact keys. Can the application safely assume every run returns parsable Python or JSON without extra text?

In [ ]:
print(ask([{"role": "user", "content": (
    "Explain an AI agent as a dictionary with exactly the keys definition, example, and limitation."
)}]))

## Exercise and checkpoint

Create a system message for your engineering discipline requiring a beginner explanation, example, limitation, and at most 150 words. Test two questions.

Instructions shape output but are not a software contract. Next we add schema-constrained output and validation.

## Your turn

Change one instruction at a time and record which behavior changes.

## Recap

Configuration shapes generation but does not guarantee truth or safety.
